In [1]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm

# Get XML from Pubmed

## Functions

In [2]:
def search_pubmed(term, num_max_results):
    url = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi'
    params = {'db' : 'pubmed', 
              'term' : term, 
              'retmode' : 'xml', 
              'retmax' : num_max_results} # retmax is max results
    response = requests.get(url, params=params)
    return response.content

In [3]:
def get_pmids_query(pubmed_response_raw):
    root = ET.fromstring(pubmed_response_raw)
    pmids = [id_elem.text for id_elem in root.findall(".//Id")]
    return pmids

In [4]:
def fetch_abstracts(pmids):
    url = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'
    string_of_ids = ','.join(pmids)
    params = {'db' : 'pubmed', 'id' : string_of_ids, 'retmode' : 'xml', 'rettype' : 'abstract'}
    response = requests.get(url, params=params)
    return response.content

## Work

In [5]:
# pubmed parameters
term = 'autism'
print(term)
num_max_results = 3000
print(num_max_results)
batch_size = 300
print(batch_size)

autism
3000
300


In [6]:
# get PMIDs
print(datetime.now())
pmids = get_pmids_query(search_pubmed(term, num_max_results))
print(datetime.now())

2025-09-14 00:30:14.135001
2025-09-14 00:30:15.543953


In [7]:
# in batches, get abstract XML
print(datetime.now())
pmids_batches = [pmids[i:i+batch_size] for i in range(0, len(pmids), batch_size)]
xml_raw_batches = [fetch_abstracts(batch) for batch in tqdm(pmids_batches)]
xml_parsed_batches = [ET.fromstring(batch) for batch in xml_raw_batches]
print(datetime.now())

2025-09-14 00:30:15.550352


  0%|          | 0/10 [00:00<?, ?it/s]

2025-09-14 00:30:41.720542


# Parse XML in batches

## Functions

In [8]:
def get_pmids(xml_parsed):
    pmids = []
    for item in xml_parsed:
        try:
            pmid = item.find(".//PMID").text
        except:
            pmid = ''
        pmids.append(pmid)
    return pmids

In [9]:
def get_elocationids(xml_parsed):
    elocationids = []
    for item in xml_parsed:
        try:
            elocationid_text = item.find(".//Article").find(".//ELocationID").text
            elocationid_type = item.find(".//Article").find(".//ELocationID").attrib.get('EIdType')
            elocationid_full = elocationid_type+': '+elocationid_text
        except:
            elocationid_full = ''
        elocationids.append(elocationid_full)
    return elocationids

In [10]:
def get_titles(xml_parsed):
    titles = []
    for item in xml_parsed:
        try:
            title = item.find(".//Article").find(".//ArticleTitle").text
        except:
            title = ''
        titles.append(title)
    return titles

In [11]:
def get_journals(xml_parsed):
    journals = []
    for item in xml_parsed:
        try:
            journal = item.find(".//Article").find(".//Journal").find(".//Title").text
        except:
            journal = ''
        journals.append(journal)
    return journals

In [12]:
def get_years(xml_parsed):
    years = []
    for item in xml_parsed:
        try:
            year = item.find(".//Article").find(".//Journal").find(".//JournalIssue").find(".//PubDate").find(".//Year").text
        except:
            year = ''
        years.append(year)
    return years

In [13]:
def get_authors(xml_parsed):
    authors = []
    for item in xml_parsed:
        try:
            auth_list = item.find(".//Article").find(".//AuthorList").findall(".//Author")
            name_list = [author.find(".//ForeName").text+' '+author.find(".//LastName").text for author in auth_list]
            authors.append(', '.join(name_list))
        except:
            authors.append('')
    return authors

In [14]:
def get_affiliations(xml_parsed):
    affiliations = []
    for item in xml_parsed:
        try:
            auth_list = item.find(".//Article").find(".//AuthorList").findall(".//Author")
            affiliation_list = [author.find(".//AffiliationInfo").find(".//Affiliation").text for author in auth_list]
            affiliations.append(' '.join(list(set(affiliation_list))))
        except:
            affiliations.append('')
    return affiliations

In [15]:
def get_abstracts(xml_parsed):
    abstracts = []
    for item in xml_parsed:
        try:
            abstract = item.find(".//Article").find(".//Abstract").find(".//AbstractText").text
        except:
            abstract = ''
        abstracts.append(abstract)
    return abstracts

## Work

In [16]:
# loop over batches, then loop over fields
column_name_to_getter = {'pmid' : get_pmids, 'elocationid' : get_elocationids, 
                        'title' : get_titles, 'journal' : get_journals, 'year' : get_years, 
                        'author' : get_authors, 'affiliation' : get_affiliations, 
                        'abstract' : get_abstracts}
df_batches = [pd.DataFrame({column_name : getter(batch) \
for (column_name, getter) in column_name_to_getter.items()}) \
for batch in xml_parsed_batches]

In [17]:
# put together
df = pd.concat(df_batches)
df

,pmid,elocationid,title,journal,year,author,affiliation,abstract
0,40944767,doi: 10.1007/s10803-025-07000-w,Psychometric Properties of the Social Responsi...,Journal of autism and developmental disorders,2025,"Fátima El-Bouhali-Abdellaoui, Núria Voltas, Pa...",Research Group on Nutrition and Mental Health ...,Earlier identification of autistic traits is c...
1,40944766,doi: 10.1007/s10803-025-07039-9,Exploring the Relationships Between Theory of ...,Journal of autism and developmental disorders,2025,"Jiaxi Li, Kathy Kar-Man Shum","Department of Psychology, The University of Ho...",This study examined friendship quality and the...
2,40944641,doi: 10.1002/hbm.70351,Flexible Reconfigurations of Brain Networks Du...,Human brain mapping,2025,"Qianying Wu, Zhihao Zhang, Ming Hsu, Andrew S ...","Helen Wills Neuroscience Institute, University...",How do large-scale brain networks dynamically ...
3,40944186,pii: 2798,Food Selectivity in Children with Autism Spect...,Nutrients,2025,"Paolo Mirizzi, Marco Esposito, Orlando Ricciar...",FUSIS MCF (Clinic Neuroscience Research and Tr...,Food selectivity is a prevalent and challengin...
4,40944170,pii: 2781,Gut Microbiota and Autism Spectrum Disorders: ...,Nutrients,2025,"Zuzanna Lewandowska-Pietruszka, Magdalena Figl...","Poznan University of Medical Sciences, Departm...",Autism spectrum disorder (ASD) is a complex ne...
...,...,...,...,...,...,...,...,...
295,40250148,doi: 10.1016/j.yebeh.2025.110420,The prevalence of comorbidities in people with...,Epilepsy & behavior : E&B,2025,"Binx Yezhe Lin, Lisa Gong, Yifan Li, Hillary S...","Division of Addiction Science, Prevention, and...",To better understand medical comorbidity in pe...
296,40249741,pii: e3003143,Vaccines work… and do not cause autism.,PLoS biology,2025,Nonia Pariente,"Public Library of Science, San Francisco, Cali...","Vaccines have saved millions of lives, yet the..."
297,40249667,doi: 10.1080/17501911.2025.2491294,Mom genes and dad genes: genomic imprinting in...,Epigenomics,2025,"Erin M O'Leary, Paul J Bonthuis","Neuroscience Program, University of Illinois, ...",Genomic imprinting is an epigenetic phenomenon...
298,40249409,doi: 10.1007/s10803-025-06815-x,Postural Control in Children with Autism Spect...,Journal of autism and developmental disorders,2025,"L Fradet, A Benchekri, R Tisserand, J-R Cazale...","Department of Child Psychiatry, Centre Hospita...",Autistic children (AT) are known to exhibit di...


In [18]:
# look at years of data
df['year'].value_counts()

year
2025    2963
2024      18
          17
2026       2
Name: count, dtype: int64

In [19]:
# count missing in each field
[df[column_name].isna().value_counts() for column_name in df.columns]

[pmid
 False    3000
 Name: count, dtype: int64,
 elocationid
 False    3000
 Name: count, dtype: int64,
 title
 False    2989
 True       11
 Name: count, dtype: int64,
 journal
 False    3000
 Name: count, dtype: int64,
 year
 False    3000
 Name: count, dtype: int64,
 author
 False    3000
 Name: count, dtype: int64,
 affiliation
 False    3000
 Name: count, dtype: int64,
 abstract
 False    2887
 True      113
 Name: count, dtype: int64]

# CSV file

In [20]:
# write CSV file
df.to_csv('../data/data.csv', sep='\t', index=False) # use tab as separator

In [21]:
print(datetime.now())

2025-09-14 00:30:42.110018
